In [20]:
from dotenv import load_dotenv
from pathlib import Path
import os

env_path = Path.cwd().parent.parent / ".env"
load_dotenv(env_path)

api_key = os.getenv("OPENAI_API_KEY")


In [21]:
from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command
from rich import print
import uuid

In [22]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage, SystemMessage
from typing import Annotated
from operator import add
from typing import Literal

In [23]:
model = ChatOpenAI(
    model ='gpt-4.1-mini',
    api_key = api_key,
    temperature=0
)

In [27]:
class State(BaseModel):
    messages: Annotated[list[BaseMessage], add] = Field(default_factory=list)
    sender_email_address:str|None = Field(description="This field contains the email address of the sender", default=None)
    recipient_email_address:str| None = Field( description ="This field contains the email address of the reciever", default=None)
    subject:str| None = Field(description="The field contains the subject of the email", default=None)
    email_body: str | None = Field( description="This field contains the email that needs which is supposed to be sent", default=None)

In [28]:
@tool
def save_final_result(
    sender_email_address: str,
    recipient_email_address:str,
    subject:str,
    email_body: str,
    runtime: ToolRuntime,
) -> Command:
    """Save the value of appropriate attributes into the state."""
    return Command(
        update={
            "sender_email_address": sender_email_address,
            "recipient_email_address": recipient_email_address,
            "subject":subject,
            "email_body":email_body,
            "messages": [
                ToolMessage(
                    content="Final result saved.",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

In [29]:
tools = [save_final_result]
model_with_tools = model.bind_tools(tools)
tool_by_name = {tool.name: tool for tool in tools}

In [30]:
tone = "formal"
length = "moderate"
prompt_template = """
    You are a helpful email assistant. You need to write an email with the following
    specifications:

    The tone of the email should be {tone} in nature, and it should be {length} in length.
"""


In [31]:
def chat_with_model(state: State):

    messages = list(state.messages)

    user_query = input("Write your question: ")

    if not messages:

        tone: Literal["formal", "informal"] = input(
            "Please mention the tone: "
        )

        length: Literal["short", "moderate", "descriptive"] = input(
            "Please mention the length: "
        )

        system_prompt = prompt_template.format(
            tone=tone,
            length=length
        )

        messages.append(SystemMessage(content=system_prompt))

    messages.append(HumanMessage(content=user_query))

    while True:

        response = model_with_tools.invoke(messages)

        if not response.tool_calls:
            messages.append(response)
            print(response.content)
            break

        messages.append(response)

        for tool_call in response.tool_calls:

            selected_tool = tool_by_name[tool_call["name"]]

            tool_result = selected_tool.invoke(
                tool_call["args"]
            )

            messages.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call["id"]
                )
            )

    return {"messages": messages}